# Capítulo 5 · Actividad: ¿este tumor es benigno?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/03_actividad_breast_cancer_logistica.ipynb)

Entrena una **regresión logística** que diga si un tumor de mama es **benigno o maligno**. Haz el mismo proceso que en las actividades del Titanic y de los autos: primero entiende los datos y después elige el modelo. El preprocesamiento lo decides tú.

Cada fila es una muestra. De la imagen del núcleo celular se miden 10 cosas (radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal). Cada una aparece tres veces:

| Grupo de columnas | Qué es |
|---|---|
| `mean ...` | Media de esa medida en la muestra |
| `... error` | Error estándar |
| `worst ...` | Media de los tres valores más grandes |
| `target` | **Target**: `0` = maligno, `1` = benigno |

`target` sale al revés de lo que uno esperaría: **1 es el caso benigno**.

Como mínimo, tu notebook debe incluir:

- La exploración de antes de modelar: forma de la tabla, balance de las dos clases, distribuciones, correlaciones y qué variables separan maligno de benigno.
- Una **regresión logística**. Las columnas no están en la misma escala (el área va por cientos y la suavidad por centésimas): decide si estandarizas.
- Métricas de **clasificación** con validación cruzada y en prueba. Como mínimo: accuracy, matriz de confusión y AUC. La accuracy sola engaña: hay más benignos que malignos.
- Tus **conclusiones**: qué modelo te quedas y por qué. Decir que un tumor maligno es benigno no es el mismo error al revés.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`

## Qué vamos a hacer

Esta actividad se parece a las del Titanic y de los autos, pero con una diferencia que cambia cómo se evalúa todo: aquí **no cuesta lo mismo equivocarse hacia un lado que hacia el otro**. Decirle a una persona con un tumor maligno que es benigno es mucho peor que mandar a estudios a una persona sana. Por eso, además de la accuracy, vamos a medir cuántos malignos se nos escapan.

El plan, en orden:

1. Explorar los datos y ver qué tan separadas están las dos clases.
2. Separar train y test, y no tocar el test hasta el final.
3. Medir qué columnas separan mejor, y decidir con evidencia si estandarizar.
4. Calcular a mano la regresión logística (sigmoide, log-loss y descenso por gradiente) y compararla con scikit-learn.
5. Elegir columnas y `C` con validación cruzada repetida, con una regla escrita antes de ver los resultados.
6. Elegir el umbral de decisión pensando en los dos tipos de error.
7. Evaluar una sola vez en test y medir con 20 repartos qué tan estable es todo.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn import set_config
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, log_loss, make_scorer,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Que scikit-learn devuelva DataFrames (con nombres de columna)
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

SEED = 42
np.random.seed(SEED)
colores = {"maligno": "#4c72b0", "benigno": "#dd8452"}

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

df = pd.read_csv(DATA_DIR / "breast_cancer.csv")
df.head()

## 1. Exploración de los datos

Primero miramos cómo vienen los datos. En esta sección solo describimos: las decisiones se toman después, con train.

In [ ]:
print("Filas y columnas:", df.shape)
print("Valores nulos:", df.isna().sum().sum(), "| Filas duplicadas:", df.duplicated().sum())

diagnostico = df["target"].map({0: "maligno", 1: "benigno"})
conteo = diagnostico.value_counts()
display(pd.DataFrame({"muestras": conteo, "porcentaje": (conteo / len(df) * 100).round(1)}))

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(x=diagnostico, order=["maligno", "benigno"], hue=diagnostico, palette=colores, legend=False, ax=ax)
ax.set_title("Muestras por diagnóstico")
ax.set_xlabel("")
ax.set_ylabel("Número de muestras")
plt.show()

In [ ]:
df.drop(columns="target").describe().T[["mean", "std", "min", "max"]].round(3)

**Qué vemos.** Hay 569 muestras y 30 columnas numéricas, sin nulos y sin filas duplicadas, así que no hay nada que limpiar. De las 569 muestras, 357 son benignas (62.7%) y 212 malignas (37.3%): las clases no están a la par, aunque tampoco hay un desbalance extremo.

Esto importa para la métrica. Un modelo que dijera siempre "benigno" acertaría el 62.7% de las veces sin aprender nada, así que ese número es el piso contra el que hay que comparar la accuracy.

Las columnas están en escalas muy distintas: `mean area` va de 143.5 a 2501, mientras que `mean smoothness` va de 0.053 a 0.163. Eso será importante cuando entrenemos el modelo.

In [ ]:
medidas = ["mean radius", "mean texture", "mean concave points", "mean smoothness", "mean symmetry", "mean fractal dimension"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for medida, ax in zip(medidas, axes.flat):
    sns.kdeplot(data=df, x=medida, hue=diagnostico, hue_order=["maligno", "benigno"], palette=colores, common_norm=False, fill=True, ax=ax)
    ax.set_title(medida)
plt.tight_layout()
plt.show()

**Qué vemos.** Hay medidas que separan muy bien las dos clases y otras que casi no:

- `mean radius` y `mean concave points` casi no se traslapan: los benignos se concentran en valores bajos y los malignos llegan mucho más arriba.
- `mean texture`, `mean smoothness` y `mean symmetry` se separan solo en parte: los malignos tienden a valores más altos, pero hay una zona grande donde conviven los dos grupos.
- `mean fractal dimension` casi no distingue: las dos curvas son prácticamente la misma.

Una sola medida no alcanza para todos los casos, pero la combinación de varias podría hacerlo bien.

In [ ]:
caracteristicas = df.drop(columns="target")
correlaciones = caracteristicas.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(correlaciones, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlación entre las 30 columnas")
plt.show()

pares = correlaciones.where(np.triu(np.ones(correlaciones.shape, dtype=bool), k=1)).stack()
muy_altas = pares[pares.abs() > 0.95].sort_values(ascending=False)
print("Pares de columnas con correlación mayor a 0.95:", len(muy_altas))
muy_altas.head(8).round(3)

**Qué vemos.** Hay bloques de columnas que miden casi lo mismo. El radio, el perímetro y el área de un núcleo celular crecen juntos de forma natural: `mean radius` y `mean perimeter` tienen una correlación de 0.998. En total hay 15 pares de columnas con correlación mayor a 0.95, y también aparecen entre las versiones `mean` y `worst` de la misma medida (por ejemplo `mean perimeter` con `worst perimeter`, 0.970).

Esto tiene una consecuencia para la regresión logística: cuando varias columnas dicen lo mismo, el modelo puede repartir el peso entre ellas de muchas maneras casi equivalentes. Las predicciones salen bien, pero los coeficientes individuales se vuelven difíciles de leer. Lo vamos a ver en la sección 8.

## 2. Train y test

Separamos antes de decidir cualquier cosa que aprenda de los datos: qué columnas usar, si estandarizar y qué hiperparámetros elegir. Usamos 80% para train y 20% para test, con `stratify` para conservar la proporción de malignos y benignos. El test no se toca hasta la sección 9.

Para comparar usamos validación cruzada estratificada de 5 partes. Para las decisiones finas la repetimos 10 veces (50 evaluaciones), porque con 455 muestras en train una sola validación cruzada cambia según cómo caigan los casos en cada parte.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Benignos en train:", (y_train == 1).sum(), "| malignos en train:", (y_train == 0).sum())
print("Benignos en test: ", (y_test == 1).sum(), "| malignos en test: ", (y_test == 0).sum())

Train queda con 455 muestras (285 benignas y 170 malignas) y test con 114 (72 benignas y 42 malignas). El detalle a tener presente: **el test solo tiene 42 casos malignos**. Cada maligno que se escape en test mueve el recall del maligno en casi 2.4 puntos porcentuales, así que las conclusiones sobre ese error van a ser poco precisas. Por eso en la sección 10 repetimos todo con otros repartos.

## 3. ¿Qué variables separan maligno de benigno?

Para cada columna calculamos su AUC por sí sola, solo con train: ¿qué tan bien ordena a los casos malignos y benignos, usando únicamente esa medida? Un AUC de 0.5 es igual que tirar una moneda y de 1.0 es separación perfecta. Como una medida puede subir o bajar con la malignidad, nos quedamos con la mayor entre el AUC y su complemento.

In [ ]:
auc_por_columna = pd.Series({col: roc_auc_score(y_train, X_train[col]) for col in X_train.columns})
auc_por_columna = np.maximum(auc_por_columna, 1 - auc_por_columna).sort_values(ascending=False)

display(auc_por_columna.head(10).round(3).rename("AUC de la columna sola"))
display(auc_por_columna.tail(5).round(3).rename("AUC de la columna sola"))

mejores = list(auc_por_columna.index[:4])
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for col, ax in zip(mejores, axes):
    sns.boxplot(x=y_train.map({0: "maligno", 1: "benigno"}), y=X_train[col], order=["maligno", "benigno"],
                hue=y_train.map({0: "maligno", 1: "benigno"}), palette=colores, legend=False, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

**Qué vemos.** Las columnas que mejor separan son las de tamaño y forma del peor caso de la muestra: `worst perimeter` (0.976), `worst radius` (0.972), `worst area` (0.971) y `worst concave points` (0.967). Con una sola de ellas ya se ordenan bien casi todos los casos.

En el otro extremo hay columnas que casi no sirven por sí solas: `mean fractal dimension` (0.508) y `texture error` (0.505) están prácticamente en 0.5, igual que tirar una moneda. Esto coincide con lo que vimos en las distribuciones.

Hay que tomarlo con cuidado: que una columna no sirva sola no quiere decir que no aporte nada combinada con otras. Por eso no descartamos columnas todavía, solo tenemos una idea de dónde está la señal.

## 4. ¿Estandarizamos?

La regresión logística usa los números tal como llegan: si el área va por cientos y la suavidad por centésimas, el algoritmo que busca los coeficientes lo tiene mucho más difícil. Lo comprobamos con tres modelos: sin escalar con las 100 iteraciones que trae `LogisticRegression` por defecto, sin escalar pero con 10,000 iteraciones, y con `StandardScaler` dentro de un Pipeline. Contamos también cuántas veces `scikit-learn` avisa que el modelo no terminó de converger.

In [ ]:
modelos_escalado = {
    "Sin escalar (100 iteraciones, por defecto)": LogisticRegression(),
    "Sin escalar (10,000 iteraciones)": LogisticRegression(max_iter=10000),
    "Con StandardScaler": Pipeline([("escalador", StandardScaler()), ("logistica", LogisticRegression())]),
}

filas = []
for nombre, modelo in modelos_escalado.items():
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        res = cross_validate(modelo, X_train, y_train, cv=cv_repetida, scoring={"accuracy": "accuracy", "auc": "roc_auc"})
    filas.append({
        "modelo": nombre,
        "accuracy": res["test_accuracy"].mean(),
        "AUC": res["test_auc"].mean(),
        "desviación del AUC": res["test_auc"].std(),
        "avisos de no convergencia (de 50)": sum(issubclass(a.category, ConvergenceWarning) for a in avisos),
    })

pd.DataFrame(filas).set_index("modelo").round(4)

**Qué vemos.** Estandarizar sí hace una diferencia clara:

- Sin escalar y con las 100 iteraciones por defecto, el modelo **no termina de converger en ninguna de las 50 evaluaciones** (50 avisos) y queda con accuracy 0.9422 y AUC 0.9887.
- Sin escalar pero con 10,000 iteraciones converge (0 avisos), aunque queda en accuracy 0.9501 y AUC 0.9907. Con tiempo de sobra llega a una solución, pero sigue por debajo.
- Con `StandardScaler` converge sin avisos y llega a accuracy 0.9780 y AUC 0.9954, además de tener la menor variación entre evaluaciones (0.0058 de desviación del AUC).

Es decir, escalar no solo acelera: aun cuando el modelo sin escalar converge, el resultado es peor. La explicación más probable es que la regularización L2 castiga el tamaño de los coeficientes, y ese tamaño depende de la escala de cada columna; con columnas estandarizadas el castigo es parejo para todas.

**Decisión:** estandarizamos, siempre dentro de un `Pipeline` para que el escalador aprenda solo con la parte de entrenamiento de cada partición y no se filtre información del test.

## 5. Cómo funciona por dentro

Antes de elegir columnas y parámetros, calculamos a mano lo que hace `LogisticRegression` y lo comparamos con scikit-learn. Usamos las 30 columnas estandarizadas.

### 5.1 De los coeficientes a la probabilidad

Un modelo lineal calcula una puntuación $z = b + w_1 x_1 + \dots + w_k x_k$. La función sigmoide la convierte en una probabilidad entre 0 y 1: $p = \dfrac{1}{1 + e^{-z}}$. Aquí $p$ es la probabilidad de que el tumor sea **benigno** (`target = 1`). Si $p \geq 0.5$, el modelo dice benigno.

In [ ]:
escalador = StandardScaler().fit(X_train)
X_train_esc = escalador.transform(X_train)
X_test_esc = escalador.transform(X_test)

logistica = LogisticRegression(C=1.0, max_iter=20000, tol=1e-10).fit(X_train_esc, y_train)

paciente = X_train_esc.iloc[[0]]
z = logistica.intercept_[0] + (paciente.to_numpy()[0] * logistica.coef_[0]).sum()
p_a_mano = 1 / (1 + np.exp(-z))

print(f"Puntuación z = {z:.4f}")
print(f"Probabilidad de benigno a mano:           {p_a_mano:.6f}")
print(f"Probabilidad de benigno con predict_proba: {logistica.predict_proba(paciente)[0, 1]:.6f}")
print("Diagnóstico real de esa muestra:", {0: "maligno", 1: "benigno"}[int(y_train.iloc[0])])

z_valores = np.linspace(-8, 8, 200)
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(z_valores, 1 / (1 + np.exp(-z_valores)))
ax.axhline(0.5, color="gray", linestyle="--")
ax.axvline(0, color="gray", linestyle="--")
ax.set_xlabel("Puntuación z")
ax.set_ylabel("Probabilidad de benigno")
ax.set_title("La función sigmoide")
plt.show()

### 5.2 El costo: log-loss

Para saber qué tan buenos son unos coeficientes se usa la **log-loss** (o entropía cruzada). Para cada muestra castiga más cuanto más seguro estaba el modelo de algo que resultó falso: $-\big[y \log p + (1 - y) \log (1 - p)\big]$. El costo es el promedio sobre las muestras. scikit-learn agrega una penalización $\dfrac{1}{2 C n}\sum w_j^2$ (regularización L2), que encoge los coeficientes; el intercepto no se penaliza. Con $C = 1$ y $n = 455$ muestras:

In [ ]:
probabilidades = logistica.predict_proba(X_train_esc)[:, 1]
log_loss_a_mano = -np.mean(y_train * np.log(probabilidades) + (1 - y_train) * np.log(1 - probabilidades))

print(f"Log-loss a mano:            {log_loss_a_mano:.6f}")
print(f"Log-loss de scikit-learn:   {log_loss(y_train, probabilidades):.6f}")

penalizacion = (logistica.coef_[0] ** 2).sum() / (2 * 1.0 * len(y_train))
print(f"Costo con la penalización L2: {log_loss_a_mano + penalizacion:.6f}")

### 5.3 Encontrar los coeficientes con descenso por gradiente

La regresión lineal tiene una fórmula cerrada. La logística no: los coeficientes se encuentran dando pasos cuesta abajo en el costo, igual que vimos en la lectura de descenso por gradiente. Con $p_i$ la probabilidad de cada muestra, el gradiente es:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{n} X^T (\mathbf{p} - \mathbf{y}) + \frac{\mathbf{w}}{C n} \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum (p_i - y_i)$$

y cada paso es $\mathbf{w} \leftarrow \mathbf{w} - \alpha \, \partial J / \partial \mathbf{w}$ (lo mismo con $b$). Lo escribimos con `numpy`, empezando de ceros, con tasa de aprendizaje $\alpha = 0.5$, y comparamos con los coeficientes de scikit-learn.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


X_np = X_train_esc.to_numpy()
y_np = y_train.to_numpy().astype(float)
n, k = X_np.shape
C_logistica = 1.0
tasa = 0.5

w = np.zeros(k)
b = 0.0
costos = []
filas = []
for paso in range(1, 20001):
    p = sigmoide(X_np @ w + b)
    costo = -np.mean(y_np * np.log(p) + (1 - y_np) * np.log(1 - p)) + (w @ w) / (2 * C_logistica * n)
    costos.append(costo)
    grad_w = X_np.T @ (p - y_np) / n + w / (C_logistica * n)
    grad_b = np.mean(p - y_np)
    w = w - tasa * grad_w
    b = b - tasa * grad_b
    if paso in (10, 100, 1000, 5000, 20000):
        filas.append({"pasos": paso, "costo": costo, "diferencia máxima en coeficientes": np.abs(w - logistica.coef_[0]).max()})

display(pd.DataFrame(filas).set_index("pasos"))
print("Diferencia en el intercepto tras 20,000 pasos:", abs(b - logistica.intercept_[0]))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(np.arange(1, 301), costos[:300])
ax.set_xlabel("Paso")
ax.set_ylabel("Costo")
ax.set_title("El costo baja en los primeros 300 pasos")
plt.show()

**Qué vemos.**

- La probabilidad calculada a mano con la sigmoide (0.999950) es idéntica a la de `predict_proba`. El modelo le da 0.99995 de probabilidad de ser benigno a esa muestra, y en efecto es benigna. En la gráfica se ve cómo la sigmoide convierte cualquier puntuación en un número entre 0 y 1: cuando $z$ es 0 la probabilidad es 0.5.
- La log-loss a mano (0.051354) coincide con la de scikit-learn. Con la penalización L2 el costo total es 0.065489.
- El descenso por gradiente que escribimos con `numpy` llega al mismo lugar: después de 5,000 pasos el costo ya es 0.065489, y después de 20,000 la diferencia máxima con los coeficientes de scikit-learn es de 6.4e-07. O sea, scikit-learn no hace nada mágico: minimiza este mismo costo.

Un detalle interesante de la tabla: a los 1,000 pasos el costo (0.065576) ya está casi en su mínimo, pero los coeficientes todavía difieren hasta 0.13 de los finales. Esto pasa porque las columnas están muy correlacionadas (sección 1): hay muchas combinaciones de coeficientes que dan casi el mismo costo, y el descenso por gradiente tarda en acomodarse en la mejor. Para predecir basta con estar cerca del mínimo, pero los coeficientes sueltos son menos confiables.

## 6. ¿Qué columnas y cuánta regularización?

Las 30 columnas están muy relacionadas entre sí (sección 1). Probamos cinco conjuntos de columnas y cinco valores de `C`, el parámetro que controla la regularización: un `C` chico encoge mucho los coeficientes y uno grande casi no los penaliza. Medimos con la validación repetida (50 evaluaciones), solo con train:

- **accuracy** y **AUC**, que ya conocemos.
- **recall del maligno**: de los tumores malignos, qué porcentaje detecta el modelo.
- **precision del maligno**: de los tumores que el modelo llama malignos, qué porcentaje lo es de verdad.

El AUC es lo que usamos para elegir. La regla se escribe antes de mirar el resultado: si varias combinaciones quedan a menos de 0.005 de AUC de la mejor, nos quedamos con la que usa menos columnas, y si empatan en eso, con la de mayor AUC.

In [ ]:
todas = list(X_train.columns)
conjuntos = {
    "todas (30)": todas,
    "mean y worst (20)": [c for c in todas if c.startswith("mean ") or c.startswith("worst ")],
    "solo worst (10)": [c for c in todas if c.startswith("worst ")],
    "solo mean (10)": [c for c in todas if c.startswith("mean ")],
    "solo error (10)": [c for c in todas if c.endswith(" error")],
}
valores_C = [0.01, 0.1, 1, 10, 100]

evaluacion = {
    "accuracy": "accuracy",
    "auc": "roc_auc",
    "recall_maligno": make_scorer(recall_score, pos_label=0),
    "precision_maligno": make_scorer(precision_score, pos_label=0, zero_division=0),
}


def armar_modelo(C=1.0):
    return Pipeline([("escalador", StandardScaler()), ("logistica", LogisticRegression(C=C, max_iter=5000))])


filas = []
for nombre, columnas in conjuntos.items():
    for C in valores_C:
        res = cross_validate(armar_modelo(C), X_train[columnas], y_train, cv=cv_repetida, scoring=evaluacion)
        filas.append({
            "columnas": nombre, "n columnas": len(columnas), "C": C,
            "accuracy": res["test_accuracy"].mean(), "AUC": res["test_auc"].mean(),
            "recall del maligno": res["test_recall_maligno"].mean(),
            "precision del maligno": res["test_precision_maligno"].mean(),
        })
tabla_modelos = pd.DataFrame(filas)

display(tabla_modelos.sort_values("AUC", ascending=False).head(10).round(4).reset_index(drop=True))
print("Mejor combinación de cada conjunto de columnas:")
display(tabla_modelos.loc[tabla_modelos.groupby("columnas")["AUC"].idxmax()].sort_values("AUC", ascending=False).round(4).reset_index(drop=True))

MARGEN_AUC = 0.005
mejor_auc = tabla_modelos["AUC"].max()
aceptables = tabla_modelos[tabla_modelos["AUC"] >= mejor_auc - MARGEN_AUC]
elegido = aceptables.sort_values(["n columnas", "AUC"], ascending=[True, False]).iloc[0]

columnas_finales = conjuntos[elegido["columnas"]]
C_final = float(elegido["C"])
print("Mejor AUC:", round(mejor_auc, 4), "| combinaciones a menos de", MARGEN_AUC, ":", len(aceptables))
print("Elegido:", elegido["columnas"], "con C =", C_final, "| AUC =", round(elegido["AUC"], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for nombre in conjuntos:
    parte = tabla_modelos[tabla_modelos["columnas"] == nombre]
    axes[0].plot(parte["C"], parte["AUC"], marker="o", label=nombre)
    axes[1].plot(parte["C"], parte["recall del maligno"], marker="o", label=nombre)
for ax, titulo in zip(axes, ["AUC", "Recall del maligno"]):
    ax.set_xscale("log")
    ax.set_xlabel("C (escala logarítmica)")
    ax.set_ylabel(titulo)
    ax.set_title(f"{titulo} según C y las columnas usadas")
axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

**Qué vemos.**

- La mejor AUC es 0.9954 y la alcanzan las 30 columnas con `C = 0.1` y con `C = 1`. Pero hay 11 combinaciones a menos de 0.005 de ese valor, y la diferencia entre ellas es más chica que la variación entre evaluaciones (alrededor de 0.006 de desviación en el AUC por partición).
- Siguiendo la regla escrita antes, se elige **solo `worst` (10 columnas) con `C = 10`**: AUC 0.9921, accuracy 0.9802, recall del maligno 0.9647 y precision del maligno 0.9828. Se queda con 10 columnas en lugar de 30 y pierde 0.0033 de AUC, una diferencia que no podemos distinguir del ruido.
- En la gráfica de la izquierda, un `C` muy chico (0.01) hace peor a todos los modelos: la regularización es tan fuerte que el modelo se queda corto. En el otro extremo, las 30 columnas bajan su AUC cuando `C` sube a 10 y a 100, que es la señal de que empiezan a sobreajustar.
- Las columnas `error` por sí solas son claramente peores (su mejor combinación llega a AUC 0.9530 y recall del maligno 0.8006). Aportan poco.
- El recall del maligno (qué tan bien detectamos los malignos) se queda alrededor de 0.96 en los mejores modelos con el umbral de 0.5. Eso es 1 de cada 25 malignos que no detectamos, y es lo que atacamos en la sección siguiente.

Una nota sobre la honestidad de la elección: con 50 evaluaciones y diferencias de milésimas, no podemos afirmar que las 10 columnas `worst` sean mejores que las 30; solo que no son peores, y un modelo con menos columnas es más fácil de explicar.

## 7. El umbral: dos errores que no cuestan lo mismo

Con $p \geq 0.5$ el modelo dice "benigno". Pero equivocarse en un sentido y en el otro no es igual:

- **Maligno clasificado como benigno** (falso negativo del maligno): el caso grave. Una persona con cáncer se va sin tratamiento.
- **Benigno clasificado como maligno**: una falsa alarma. Cuesta estudios y angustia, pero se descubre con más pruebas.

Subir el umbral (pedir $p \geq 0.8$ para decir "benigno") hace que el modelo sea más cauteloso: deja pasar menos malignos, a cambio de más falsas alarmas. Para elegir el umbral sin usar el test, calculamos probabilidades **fuera de muestra** en train: con `cross_val_predict`, cada muestra se predice con un modelo que no la vio. La regla, escrita antes de mirar la tabla: nos quedamos con el umbral de mayor accuracy entre los que detectan al menos el 98% de los malignos.

In [ ]:
modelo_final = armar_modelo(C_final)
prob_fuera_de_muestra = cross_val_predict(modelo_final, X_train[columnas_finales], y_train, cv=cv, method="predict_proba")[:, 1]

umbrales = np.round(np.arange(0.30, 0.96, 0.05), 2)
filas = []
for umbral in umbrales:
    pred = (prob_fuera_de_muestra >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, pred).ravel()      # clase 0 = maligno, clase 1 = benigno
    filas.append({
        "umbral": umbral,
        "malignos que se escapan (FN)": fp,
        "falsas alarmas": fn,
        "recall del maligno": tn / (tn + fp),
        "accuracy": (tn + tp) / len(y_train),
    })
tabla_umbral = pd.DataFrame(filas).set_index("umbral")
display(tabla_umbral.round(3))

cumplen = tabla_umbral[tabla_umbral["recall del maligno"] >= 0.98]
umbral_elegido = float(cumplen["accuracy"].idxmax()) if len(cumplen) else float(tabla_umbral["recall del maligno"].idxmax())
print("Umbral elegido:", umbral_elegido)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tabla_umbral.index, tabla_umbral["malignos que se escapan (FN)"], marker="o", label="Malignos que se escapan")
ax.plot(tabla_umbral.index, tabla_umbral["falsas alarmas"], marker="o", label="Falsas alarmas")
ax.axvline(umbral_elegido, color="gray", linestyle="--", label=f"Umbral elegido ({umbral_elegido})")
ax.set_xlabel("Umbral de probabilidad de benigno")
ax.set_ylabel("Casos en train (fuera de muestra)")
ax.set_title("Más umbral: menos malignos que se escapan, más falsas alarmas")
ax.legend()
plt.show()

**Qué vemos.** Con probabilidades fuera de muestra en train:

- Con el umbral de 0.5, se escapan 6 malignos de 170 (recall del maligno 0.965) y hay 4 falsas alarmas.
- Subir el umbral a 0.7 baja los malignos que se escapan a 3 (recall 0.982) a cambio de 8 falsas alarmas, y la accuracy casi no cambia (0.976 contra 0.978).
- De 0.75 en adelante los malignos que se escapan se quedan en 2, y las falsas alarmas se disparan (de 10 a 37). O sea, hay 2 tumores malignos que el modelo califica con al menos 0.95 de probabilidad de ser benignos, y ningún umbral los atrapa sin marcar casi todo como maligno.

Con la regla escrita antes (mayor accuracy entre los umbrales que detectan al menos 98% de los malignos), el umbral elegido es **0.7**.

Hay que entender lo que esto significa: movimos el umbral porque el costo de los errores es asimétrico, no porque el modelo sea mejor. La AUC sigue siendo la misma; solo cambiamos el punto de operación del modelo.

## 8. Modelo final

Entrenamos con todo train el modelo que salió de la sección 6, con las columnas y el `C` elegidos, y fijamos el umbral de la sección 7. Mostramos sus coeficientes: como las columnas están estandarizadas, un coeficiente positivo sube la probabilidad de benigno y uno negativo la baja, y su tamaño se puede comparar entre columnas.

In [ ]:
modelo_final.fit(X_train[columnas_finales], y_train)
coeficientes = pd.Series(modelo_final.named_steps["logistica"].coef_[0], index=columnas_finales).sort_values(key=abs, ascending=False)

print("Intercepto:", round(modelo_final.named_steps["logistica"].intercept_[0], 3))
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=coeficientes.values, y=coeficientes.index, hue=coeficientes.index, palette="coolwarm_r", legend=False, ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Coeficientes (positivo: más probable benigno)")
plt.show()
coeficientes.round(3)

Los coeficientes más grandes son `worst area` (-3.181), `worst radius` (-2.274), `worst concave points` (-1.846) y `worst texture` (-1.735): valores altos de estas medidas bajan la probabilidad de benigno, es decir, sugieren un tumor maligno. Esto coincide con lo que vimos en la exploración.

Pero hay una advertencia: `worst compactness` sale con coeficiente **positivo** (1.373), como si más compacidad indicara benigno, y en la exploración los malignos tenían valores más altos. Y `worst area` y `worst radius` aparecen los dos con coeficientes grandes aunque miden casi lo mismo. Esto es el efecto de la correlación entre columnas: el modelo reparte y compensa los pesos entre columnas que se parecen. Por eso **el modelo sirve para predecir, pero no hay que interpretar cada coeficiente por separado** como "el efecto de esa medida".

## 9. Evaluación en test

Esta es la primera vez que usamos el test. La decisión del modelo y del umbral ya está tomada, así que estos números no la cambian. Comparamos con un modelo que no aprende nada: decir siempre "benigno".

In [ ]:
prob_test = modelo_final.predict_proba(X_test[columnas_finales])[:, 1]
n_malignos_test = int((y_test == 0).sum())

resultados = {}
for nombre, umbral in [("Umbral 0.5", 0.5), (f"Umbral {umbral_elegido}", umbral_elegido)]:
    pred = (prob_test >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    resultados[nombre] = {
        "accuracy": (tn + tp) / len(y_test),
        "malignos que se escapan": int(fp),
        "falsas alarmas": int(fn),
        "recall del maligno": tn / (tn + fp),
        "precision del maligno": tn / (tn + fn) if (tn + fn) else np.nan,
    }

siempre_benigno = {"accuracy": (y_test == 1).mean(), "malignos que se escapan": n_malignos_test, "falsas alarmas": 0,
                   "recall del maligno": 0.0, "precision del maligno": np.nan}
resultados["Siempre 'benigno'"] = siempre_benigno

tabla_test = pd.DataFrame(resultados).T
tabla_test["AUC"] = [roc_auc_score(y_test, prob_test)] * 2 + [0.5]
tabla_test = tabla_test.astype({"malignos que se escapan": int, "falsas alarmas": int})
tabla_test.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (nombre, umbral) in zip(axes[:2], [("Umbral 0.5", 0.5), (f"Umbral {umbral_elegido}", umbral_elegido)]):
    pred = (prob_test >= umbral).astype(int)
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["maligno", "benigno"], yticklabels=["maligno", "benigno"], ax=ax)
    ax.set_title(nombre)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")

fpr, tpr, _ = roc_curve(y_test, prob_test)
axes[2].plot(fpr, tpr, label=f"Modelo final (AUC = {roc_auc_score(y_test, prob_test):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", label="Azar")
for nombre, umbral in [("0.5", 0.5), (str(umbral_elegido), umbral_elegido)]:
    pred = (prob_test >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    axes[2].plot(fp / (tn + fp), tp / (tp + fn), "o", markersize=9, label=f"umbral {nombre}")
axes[2].set_xlabel("Malignos clasificados como benignos")
axes[2].set_ylabel("Benignos clasificados como benignos")
axes[2].set_title("Curva ROC en test")
axes[2].legend(loc="lower right")
plt.tight_layout()
plt.show()

print(classification_report(y_test, (prob_test >= umbral_elegido).astype(int), target_names=["maligno", "benigno"], digits=3))
recall_test = tabla_test.loc[f"Umbral {umbral_elegido}", "recall del maligno"]
print(f"Error estándar aproximado del recall del maligno con {n_malignos_test} malignos: {np.sqrt(recall_test * (1 - recall_test) / n_malignos_test):.3f}")

**Qué vemos.**

- El modelo final con umbral 0.5 acierta el 95.6% de los casos de test (AUC 0.996) y se le escapa **1 maligno de 42** (recall del maligno 0.976), con 4 falsas alarmas.
- Con el umbral elegido de 0.7 también se escapa 1 maligno de 42, y las falsas alarmas suben de 4 a 7 (accuracy 0.930). Es decir, **en este test en particular, subir el umbral no ayudó**: el maligno que se escapa ya tenía una probabilidad de benigno de al menos 0.7 y el cambio solo añadió falsas alarmas. Lo reportamos así tal cual: el umbral se eligió con train y en este reparto de test no se notó el beneficio.
- El modelo que dice siempre "benigno" tiene accuracy 0.632 y deja pasar a los 42 malignos. Nuestro modelo está muy por encima, y ese 0.632 es el piso contra el que hay que comparar cualquier accuracy.

Sobre la incertidumbre: con 42 malignos en test, el recall del maligno tiene un error estándar de alrededor de 0.024, así que 0.976 podría ser fácilmente 0.95 o 1.0 en otro reparto. Un solo test no basta para decidir entre los dos umbrales; para eso está la sección 10.

## 10. ¿Qué tan estable es el resultado?

El test tiene 114 muestras y solo 42 son malignas. Para ver cuánto cambia el resultado con otro reparto, repetimos el proceso con 20 repartos distintos (`random_state` de 0 a 19). En cada uno separamos, elegimos `C` solo con el train de ese reparto (con una rejilla chica), y medimos en su test con el umbral 0.5 y con el umbral elegido. Las columnas y el umbral los dejamos fijos, así que esta comparación tiene una ventaja pequeña: esas decisiones se tomaron con el train del reparto 42. Esto solo mide la variación; no sirve para escoger el modelo.

In [ ]:
filas = []
for semilla in list(range(20)) + [SEED]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    busqueda = GridSearchCV(armar_modelo(), {"logistica__C": valores_C}, cv=cv, scoring="roc_auc")
    busqueda.fit(X_tr[columnas_finales], y_tr)
    prob = busqueda.predict_proba(X_te[columnas_finales])[:, 1]

    fila = {"reparto": semilla, "AUC": roc_auc_score(y_te, prob)}
    for nombre, umbral in [("0.5", 0.5), ("elegido", umbral_elegido)]:
        pred = (prob >= umbral).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        fila[f"accuracy ({nombre})"] = (tn + tp) / len(y_te)
        fila[f"malignos que se escapan ({nombre})"] = int(fp)
        fila[f"falsas alarmas ({nombre})"] = int(fn)
    filas.append(fila)
repartos = pd.DataFrame(filas).set_index("reparto")

otros = repartos.drop(index=SEED)    # los 20 repartos distintos al del notebook
resumen = pd.DataFrame({"media": otros.mean(), "mínimo": otros.min(), "máximo": otros.max(), "reparto 42": repartos.loc[SEED]})
display(resumen.round(3))
print("Repartos en los que se escapa al menos un maligno con el umbral 0.5:", int((otros["malignos que se escapan (0.5)"] >= 1).sum()), "de 20")
print("Repartos en los que se escapa al menos un maligno con el umbral elegido:", int((otros["malignos que se escapan (elegido)"] >= 1).sum()), "de 20")
malignos_menos = otros["malignos que se escapan (0.5)"].mean() - otros["malignos que se escapan (elegido)"].mean()
alarmas_mas = otros["falsas alarmas (elegido)"].mean() - otros["falsas alarmas (0.5)"].mean()
print(f"Falsas alarmas extra por cada maligno que se deja de escapar: {alarmas_mas / malignos_menos:.1f}")

**Qué vemos.**

- El AUC es muy estable: media 0.994, entre 0.970 y 1.000. El modelo separa bien las clases en cualquier reparto.
- Nuestro reparto 42 salió algo peor que el promedio: accuracy de 0.956 con umbral 0.5 contra una media de 0.979, y 4 falsas alarmas contra una media de 0.800.
- **Aquí sí se ve el efecto del umbral.** En promedio, con el umbral 0.5 se escapan 1.650 malignos por reparto, y con 0.7 se escapan 0.900. A cambio, las falsas alarmas suben de 0.800 a 2.700 en promedio y la accuracy baja de 0.979 a 0.968.
- Con el umbral 0.5 se escapa al menos un maligno en 19 de los 20 repartos, y con el umbral 0.7 en 14 de 20. Disminuye, pero **no desaparece**.

Entonces el resultado de la sección 9 no contradice la sección 7, solo es un reparto en el que no se notó. Con 20 repartos el beneficio de 0.7 se ve: por cada maligno que se deja de escapar aparecen alrededor de 2.5 falsas alarmas más.

## Conclusiones

**Qué modelo nos quedamos.** Una regresión logística con las 10 columnas `worst` estandarizadas (dentro de un `Pipeline`), `C = 10` y umbral de decisión de 0.7. En test acierta el 93.0% de los casos con el umbral 0.7 (95.6% con 0.5), tiene AUC 0.996 y detecta 41 de los 42 malignos.

**Decisiones y evidencia**

| Decisión | Qué se hizo | Evidencia |
|---|---|---|
| Estandarizar | Sí, con `StandardScaler` en un Pipeline | Sin escalar no converge con 100 iteraciones (50 avisos de 50) y queda en AUC 0.9887; escalado llega a 0.9954 |
| Columnas | Solo las 10 `worst` | A 0.0033 de AUC de las 30 columnas, dentro del ruido; regla: menos columnas si el AUC queda a menos de 0.005 |
| Regularización | `C = 10` | Elegido con la validación repetida de train (50 evaluaciones) |
| Umbral | 0.7 en vez de 0.5 | Con probabilidades fuera de muestra en train: de 6 a 3 malignos que se escapan, de 4 a 8 falsas alarmas |
| Método de evaluación | Test usado una sola vez | Todas las decisiones se tomaron con train |

**Lo que mostró la evidencia**

1. La regresión logística, con los datos bien preparados, separa muy bien tumores benignos y malignos: AUC de 0.996 en test y de 0.994 en promedio en 20 repartos.
2. La accuracy sola engaña: decir siempre "benigno" da 0.632, y los modelos buenos están entre 0.93 y 0.98. Lo que de verdad importa es cuántos malignos se escapan.
3. Mover el umbral cambia el tipo de error, no la calidad del modelo. En 20 repartos, pasar de 0.5 a 0.7 reduce los malignos que se escapan de 1.650 a 0.900 por reparto, a cambio de pasar de 0.800 a 2.700 falsas alarmas.
4. Hay malignos que ningún umbral razonable atrapa: el modelo les da al menos 0.95 de probabilidad de ser benignos. Con este modelo y estos datos no se llega a cero malignos escapados.

**Limitaciones**

- El test tiene solo 42 casos malignos: los porcentajes de recall se mueven casi 2.4 puntos por cada caso. Por eso apoyamos las conclusiones en los 20 repartos y no en uno solo.
- En la comparación de 20 repartos las columnas y el umbral se dejaron fijos con las decisiones del reparto 42, así que esa comparación es un poco optimista. Sirve para medir la variación, no para escoger el modelo.
- Los coeficientes no se pueden leer uno por uno por la correlación entre columnas (por ejemplo, el signo positivo de `worst compactness`).
- Son 569 muestras de una sola fuente, sin validación con datos de otro hospital ni otro equipo. Este es un ejercicio de aprendizaje y el modelo no se debería usar para diagnosticar a nadie.

**Qué haríamos con más tiempo:** probar otros modelos para comparar (por ejemplo, un árbol o un bosque) con la misma validación repetida, y escoger el umbral con costos reales de cada error en lugar de una regla de 98% de recall.